# Day 7 Lab 01: Spark DataFrame Basics

        **Layer:** Spark fundamentals  
        **Python reference:** `Lab_Files/labs/lab_01_spark_dataframe_basics.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Start Spark from a notebook.
- Read customers.csv with the shared explicit schema.
- Apply simple DataFrame transformations and write a preview artifact.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
LAB_FILES = HERE / "Lab_Files"
if not LAB_FILES.exists():
    LAB_FILES = HERE.parent / "Lab_Files"

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")

## 1. Import Lab Helpers

In [ ]:
from pyspark.sql import functions as F

from day7_common import OUTPUT_DIR, ensure_output_dirs, read_customers, require_source_data, spark_session, write_csv_dir

## 2. Start Spark and Validate Source Data

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook01DataFrameBasics")

## 3. Read the Customer Dimension

In [ ]:
customers = read_customers(spark)
customers.printSchema()
customers.show(10, truncate=False)

## 4. Transform for a Learner-Friendly Preview

In [ ]:
preview = (
    customers.withColumn("email", F.lower(F.trim("email")))
    .withColumn("signup_date", F.to_date("signup_date"))
    .select("customer_id", "customer_name", "email", "region", "loyalty_tier", "signup_date")
    .orderBy("customer_id")
)

## 5. Inspect and Write Output

In [ ]:
preview.show(20, truncate=False)
write_csv_dir(preview, OUTPUT_DIR / "notebook_01_customers_preview")

## 6. Checkpoint

In [ ]:
print(f"Rows read from customers.csv: {customers.count()}")
print("Concepts: SparkSession, explicit schema, lazy transformations, action via count().")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()